In [1]:
import os
from pathlib import Path
from uuid import uuid4

from langchain_core.documents import Document
from langchain_community.document_loaders import PyPDFLoader, TextLoader, UnstructuredMarkdownLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter, CharacterTextSplitter
from langchain_ollama import OllamaEmbeddings, ChatOllama
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_chroma import Chroma

from src.md_reader import PreProcMd

#Set your ollama host to the port you started the server
OLLAMA_HOST = "127.0.0.1:40063"
os.environ["OLLAMA_HOST"] = "127.0.0.1:40063"
print(OLLAMA_HOST)

# ==========================================
EMBED_MODEL = "mxbai-embed-large:latest"
LLM_MODEL = "llama3.2:3b"
# ==========================================

127.0.0.1:40063


# Building Knowledge Base

## Loading document(s)
Here we'll load just a single document with classes from the [langchain-community](https://reference.langchain.com/python/langchain-community) document loader module. In the base example, we load a pdf file, but you may load pdf, txt, or markdown files. The loaders extract and load the files as [Document objects](https://reference.langchain.com/python/langchain-core/documents) defined in the [LangChain Core](https://reference.langchain.com/python/langchain-core) Python package.

In [2]:
def load_document(fpath: str):
    ext = os.path.splitext(fpath)[-1].lower()
    if ext == ".pdf":
        #mode: 'single' -> entire doc, (default) 'page' -> 1 doc/page
        loader = PyPDFLoader(fpath, mode="page")
        docs = loader.load()
    elif ext in [".md", ".markdown"]:
        repo_path = os.path.dirname(fpath)
        splits = PreProcMd(repo_path).run()
        target = os.path.abspath(fpath)
        docs = [
            d for d in splits
            if os.path.abspath(str(d.metadata.get("source", ""))) == target
        ]
    elif ext == ".txt":
        loader = TextLoader(fpath, encoding="utf-8")
        docs = loader.load()
    else:
        raise ValueError(f"Unsupported file type: {fpath}")
    print(f"Loaded {len(docs)} docs from {fpath}")
    return docs

In [3]:
docs_dir = '/mnt/research/HPCC_AI_Workshop/use-case/data/icer_docs' #change to your project space if you've copied these data over.
file_name = 'data_machine.pdf'
file_path = os.path.join(docs_dir, file_name) #path to all doc: /mnt/ufs18/rs-007/HPCC_AI_Workshop/use-case/data
docs = load_document(file_path) #in 'page' mode number of docs here == number of pages in the actual document

Loaded 5 docs from /mnt/research/HPCC_AI_Workshop/use-case/data/icer_docs/data_machine.pdf


In [4]:
docs[0]

Document(metadata={'producer': 'pdfmake', 'creator': 'pdfmake', 'creationdate': '2025-10-24T14:13:13+00:00', 'source': '/mnt/research/HPCC_AI_Workshop/use-case/data/icer_docs/data_machine.pdf', 'total_pages': 5, 'page': 0, 'page_label': '1'}, page_content='The Data Machine\n \nICER is excited to offer a new computational resource called the “Data Machine”. Many research \nareas are now faced with large amounts of data thanks in large part to the growth of available \ndatasets. Manipulating, analyzing, and visualizing  large amounts of data requires specialized \ncomputational resources that are not typically offered by more traditional high performance \ncomputing systems. Additionally, this “data explosion” is occurring primarily in fields where \nresearch computing has historically not been widely used.\n \nTo meet these hardware and educational needs, ICER is developing the Data Machine and \nassociated outreach and training programs. Though the machine is not yet available to the \

## Splitting & Chunking Docs
We'll now take the loaded docs (list of Document objects), and split each of them into smaller chunks using the [`RecursiveCharacterTextSplitter`](https://reference.langchain.com/python/langchain-text-splitters/character/RecursiveCharacterTextSplitter) from the [LangChain Text Splitters](https://reference.langchain.com/python/langchain-text-splitters) Python package. This applies splitting recursively based on typical splitting characters (e.g., ["\n\n", "\n", " ", ""]). The method first splits by paragraphs with "\n\n" and further splits each of those with "\n" (then space and empty string). Once the doc has been fully split, it then merges the split text to define chunks with maximum length determined by our set chunk size. 

The chunk size is essentially the maximum (and targeted) size of a chunk determined by length (len function by default). The chunk overlap is also a targeted value. Overlap helps maintain context when divided between chunks.

>Note the resulting chunk sizes and overlaps are often not exact, as these methods look for clean text boundaries (e.g., paragraph, line, word).  

In [5]:
def conditional_split(docs: list[Document], chunk_size: int, chunk_overlap: int) -> list[Document]:
    """
    Conditionally apply chunking only to documents longer than chunk_size.
    Docs <= chunk_size pass through unchanged (1 chunk).
    Docs > chunk_size are split using RecursiveCharacterTextSplitter with overlap.

    Params:
        docs: Input list of Documents (typically markdown will be already structurally split).
        chunk_size: Target max characters per chunk.
        chunk_overlap: Number of overlapping characters between consecutive chunks.

    Returns:
        A list of Documents where longer inputs are expanded into multiple chunks.
    """
    splitter=RecursiveCharacterTextSplitter(chunk_size=chunk_size,chunk_overlap=chunk_overlap,add_start_index=True, separators=None, ) #default splitting characters ["\n\n", "\n", " ", ""]
    # splitter=CharacterTextSplitter(chunk_size=chunk_size,chunk_overlap=chunk_overlap,add_start_index=True,)
    out = []
    for i in docs:
        if len(i.page_content) <= chunk_size:
            out.append(i)  
        else:
            out.extend(splitter.split_documents([i]))  
    return out

In [6]:
# text_splitter = RecursiveCharacterTextSplitter(
#         chunk_size=1000, chunk_overlap=200, add_start_index=True
#     )
splits = conditional_split(docs, chunk_size=700, chunk_overlap=100)
#splits = text_splitter.split_documents(all_docs)
print(f"Split into {len(splits)} chunks.")

Split into 21 chunks.


In [7]:
splits[0].page_content[-150:]

'ere \nresearch computing has historically not been widely used.\n \nTo meet these hardware and educational needs, ICER is developing the Data Machine and'

In [8]:
splits[1].page_content[:150]

'To meet these hardware and educational needs, ICER is developing the Data Machine and \nassociated outreach and training programs. Though the machine i'

## Document Embedding
Next, we will embed the documents and put them in a vector store collection. We'll use the [LangChain Ollama embedding integration](https://reference.langchain.com/python/langchain-ollama/embeddings/OllamaEmbeddings) to embed the docs with a local embedding model from our Ollama common model space. Here, we're using a [Chroma](https://docs.trychroma.com/docs/overview/introduction) vector store with the [LangChain Chroma integration](https://docs.langchain.com/oss/python/integrations/vectorstores/chroma).

In [9]:
embeddings = OllamaEmbeddings(model=EMBED_MODEL)
vector_store = InMemoryVectorStore(embeddings)
vector_store.add_documents(splits)
print(f"{len(vector_store.store)} docs in vector store, one for each chunk.")

21 docs in vector store, one for each chunk.


### Creating the vector store
The vector store is then created with name for our docs collection, and the `OllamaEmbeddings` instance to define/configure the embedding function. Note the `OllamaEmbeddings` class is used to store configuration data of the selected embedding model, and implements the embedding function using the selected embedding model. 

> In this case the vector store is setup and running in-memory, not on disk/external db.

> Run vector_store._client.delete_collection(collection_name) to remove collection from chroma vector store

In [10]:
collection_name = "icer_docs_collection"

embeddings = OllamaEmbeddings(model=EMBED_MODEL)
vector_store = Chroma(
    collection_name=collection_name,
    embedding_function=embeddings,
)

#vector_store._client.delete_collection("icer_docs_collection")

Now we add the documents to the vector store, providing unique ids to each document. The document embeddings are included in this step.

In [11]:
%%time

uuids = [str(uuid4()) for _ in range(len(splits))]
vector_store.add_documents(documents=splits, ids=uuids) # add_documents applies the embedding function to the documents (splits)
collection = vector_store._collection
n_docs_stored = collection.count()
print(f"{n_docs_stored} docs in vector store, one for each chunk.")

21 docs in vector store, one for each chunk.
CPU times: user 22.9 ms, sys: 7.43 ms, total: 30.4 ms
Wall time: 6.69 s


In [12]:
vs_embedding_info = vector_store._collection.get(include=["embeddings"])
doc_embeddings = vs_embedding_info['embeddings']

print(f"{doc_embeddings.shape[0]} doc embeddings of size {doc_embeddings.shape[1]} in vector store: \n\n", doc_embeddings)

21 doc embeddings of size 1024 in vector store: 

 [[ 0.02548597  0.00183591 -0.02746432 ... -0.01732768 -0.00507805
   0.03609309]
 [ 0.05354257  0.02565196 -0.05158571 ... -0.01262257 -0.01461829
   0.04328546]
 [ 0.02563223 -0.00211609  0.01004239 ...  0.00018149 -0.01497238
   0.06008346]
 ...
 [ 0.06278302 -0.01110651 -0.00132954 ... -0.07038857 -0.01094996
   0.01734013]
 [ 0.05116982 -0.02099884 -0.02040707 ... -0.03487178 -0.00247381
   0.02484727]
 [ 0.02855283 -0.00057942 -0.02564297 ... -0.0227016   0.00216078
   0.03079591]]


In [13]:
print(vs_embedding_info)

{'ids': ['44fcd97c-a913-42db-81e2-aa3430be8c78', '06ee853e-59d8-424d-b31d-5764106e5f5a', 'a711d26e-c9cf-4d69-b46a-082d60a142c8', '6f883e45-4c5e-46d8-9211-58d2f40852f7', '9c84e037-3227-48ee-a3c3-cf3794ba8084', '30eb7698-fc86-46a2-928f-292aa94b1832', '8f364877-3e1d-4ea8-91ee-9a8be6496500', '009e6e6e-ab7a-46d7-a9be-a4be4109a028', '9d824a74-6d1c-43ed-be39-4d033822603b', '37d1867c-a4db-41be-9cd3-9931105ce5c6', 'bdaeb578-a53c-4ca5-892b-ec0725e45638', 'c6d3e3eb-d172-4be2-9f5d-9afb48a61eb4', 'f6cc076d-46d5-4d8e-bd7f-c20e82c60799', '70ff7f08-6e36-4602-a8b1-049f30cf0f8e', '1c7575f8-12a9-491d-b7bf-67e209977ac8', '0bb39727-b195-4c50-8dc0-64c511b3b20c', 'b38de139-38c1-436b-a67d-3d67e6baf9d2', 'aa644571-ba8f-4406-9963-088602a61bfe', 'e22b7309-baa4-4d5d-9f5a-66f683c494dd', 'cdfcb28b-779f-4437-8b1e-534d5053350e', '85894588-b05a-40c7-a3c5-77c63136e39c'], 'embeddings': array([[ 0.02548597,  0.00183591, -0.02746432, ..., -0.01732768,
        -0.00507805,  0.03609309],
       [ 0.05354257,  0.02565196, -0

In [14]:
# vector_store._collection.get(include=["embeddings"]) #include ["documents", "metadatas", "embeddings"]

Here we run a similarity search for a simple query. The query is embedded and checked against the vector store embeddings, selecting the top k docs by similarity (or distance). Another method is using [`similarity_search_with_relevance_scores`](https://reference.langchain.com/python/langchain-core/vectorstores/base/VectorStore/similarity_search_with_relevance_scores) which scores increasing with similarity.  

In [15]:
results = vector_store.similarity_search_with_score(
    "What type of GPUs are on The Data Machine?",
    k=5,
    filter={"source": file_path},
)
for res, score in results:
    print(f"* [similarity={score:3f}] {res.metadata} \n")
    print(res.page_content + "\n")

* [similarity=0.464705] {'source': '/mnt/research/HPCC_AI_Workshop/use-case/data/icer_docs/data_machine.pdf', 'start_index': 2216, 'creator': 'pdfmake', 'page': 1, 'producer': 'pdfmake', 'page_label': '2', 'creationdate': '2025-10-24T14:13:13+00:00', 'total_pages': 5} 

transferring large volumes of data. The Data Machine will have a total of 8 nodes, 4 focused on 
CPU-intensive jobs and 4 for jobs requiring GPUS.
 
The Data Machine will be connected to the existing HPCC file systems and compute resources. 
Though the CPU and GPU hardware is similar to what is offered by the current amd21 and 
amd22 clusters, what sets the Data Machine apart is the amount of memory available to CPU

* [similarity=0.483376] {'total_pages': 5, 'start_index': 531, 'creator': 'pdfmake', 'page': 2, 'source': '/mnt/research/HPCC_AI_Workshop/use-case/data/icer_docs/data_machine.pdf', 'creationdate': '2025-10-24T14:13:13+00:00', 'producer': 'pdfmake', 'page_label': '3'} 

units, making them ideal for machine l

In [16]:
results = vector_store.similarity_search_with_relevance_scores(
    "What type of GPUs are on The Data Machine?",
    k=5,
    filter={"source": file_path},
)
for res, score in results:
    print(f"* [similarity={score:3f}] {res.metadata} \n")
    print(res.page_content + "\n")

* [similarity=0.671404] {'producer': 'pdfmake', 'creator': 'pdfmake', 'page_label': '2', 'source': '/mnt/research/HPCC_AI_Workshop/use-case/data/icer_docs/data_machine.pdf', 'page': 1, 'start_index': 2216, 'creationdate': '2025-10-24T14:13:13+00:00', 'total_pages': 5} 

transferring large volumes of data. The Data Machine will have a total of 8 nodes, 4 focused on 
CPU-intensive jobs and 4 for jobs requiring GPUS.
 
The Data Machine will be connected to the existing HPCC file systems and compute resources. 
Though the CPU and GPU hardware is similar to what is offered by the current amd21 and 
amd22 clusters, what sets the Data Machine apart is the amount of memory available to CPU

* [similarity=0.658202] {'page_label': '3', 'start_index': 531, 'producer': 'pdfmake', 'total_pages': 5, 'source': '/mnt/research/HPCC_AI_Workshop/use-case/data/icer_docs/data_machine.pdf', 'creationdate': '2025-10-24T14:13:13+00:00', 'page': 2, 'creator': 'pdfmake'} 

units, making them ideal for machine l

## Building Knowledge Base
We'll now extend document embedding to a set of docs (pdfs in this example).

In [17]:
docs_repo = Path(docs_dir)
file_paths = list(docs_repo.glob("*.pdf"))
all_docs = []
for path in file_paths:
    all_docs.extend(load_document(path))

splits = conditional_split(all_docs, chunk_size=700, chunk_overlap=100)
print(f"Split into {len(splits)} chunks.")

Loaded 5 docs from /mnt/research/HPCC_AI_Workshop/use-case/data/icer_docs/data_machine.pdf
Loaded 3 docs from /mnt/research/HPCC_AI_Workshop/use-case/data/icer_docs/requesting_gpus.pdf
Loaded 3 docs from /mnt/research/HPCC_AI_Workshop/use-case/data/icer_docs/cluster_resources.pdf
Loaded 5 docs from /mnt/research/HPCC_AI_Workshop/use-case/data/icer_docs/llms_hpcc.pdf
Loaded 2 docs from /mnt/research/HPCC_AI_Workshop/use-case/data/icer_docs/gpu_resources.pdf
Loaded 2 docs from /mnt/research/HPCC_AI_Workshop/use-case/data/icer_docs/user_data_policies.pdf
Loaded 1 docs from /mnt/research/HPCC_AI_Workshop/use-case/data/icer_docs/dev_nodes.pdf
Loaded 3 docs from /mnt/research/HPCC_AI_Workshop/use-case/data/icer_docs/job_policies.pdf
Loaded 3 docs from /mnt/research/HPCC_AI_Workshop/use-case/data/icer_docs/interactive_job.pdf
Loaded 12 docs from /mnt/research/HPCC_AI_Workshop/use-case/data/icer_docs/faq.pdf
Split into 139 chunks.


In [18]:
vector_store = Chroma(
    collection_name="icer_docs_collection_full",
    embedding_function=embeddings,
    persist_directory="./chroma_icer_docs_db",
)
print(vector_store._collection.get())

{'ids': ['e9c426b6-9608-4130-8153-e54367b21d4f', '29951f21-1db2-451a-99ec-1309eaec9712', '6d572d7c-3b9d-4cef-9e38-1d58ffacefbc', '052b2362-5a5b-4510-a8cc-088ff74af7bb', 'c94d2aba-bf9b-4a9a-8667-0f89925edb60', '5ac15c5e-e5ae-435a-868d-c179b6448cbc', '727f76ed-56e0-467b-bb0a-8f8c830ffe73', 'f34cbef4-a499-4038-a945-e28823accc2d'], 'embeddings': None, 'documents': ['lab notebook LLMs\n(2025-03-05) Lab Notebook: Update: How to\nrun RAG in OpenWebUI\nThis is as a Lab Notebook which describes how to solve a specific problem at a specific time.\nPlease keep this in mind as you read and use the content. Please pay close attention to the\ndate, version information and other details.\nRetrieval-Augmented Generation (RAG) is a technique that grants generative artificial\nintelligence models information retrieval capabilities. It modifies interactions with a large\nlanguage model so that the model responds to user queries with reference to a specified set\nof documents. This allows the model to gen

In [19]:
%%time

uuids = [str(uuid4()) for _ in range(len(splits))]
vector_store.add_documents(documents=splits, ids=uuids)
collection = vector_store._collection
n_docs_stored = collection.count()
print(f"{n_docs_stored} docs in vector store, one for each chunk.")

147 docs in vector store, one for each chunk.
CPU times: user 124 ms, sys: 32.1 ms, total: 156 ms
Wall time: 50.6 s


You can now query against a larger set of documents. As you can see, it may be nice to not have to run the embeddings each time you want to query against your documents, and instead query against the built and exisisting vector store. 

In [20]:
results = vector_store.similarity_search_with_score(
    "What type of GPUs are on The Data Machine?",
    k=5,
    filter={"source": file_path},
)
for res, score in results:
    print(f"* [similarity={score:3f}] {res.metadata} \n")
    print(res.page_content + "\n")

* [similarity=0.464705] {'source': '/mnt/research/HPCC_AI_Workshop/use-case/data/icer_docs/data_machine.pdf', 'creationdate': '2025-10-24T14:13:13+00:00', 'start_index': 2216, 'producer': 'pdfmake', 'creator': 'pdfmake', 'total_pages': 5, 'page': 1, 'page_label': '2'} 

transferring large volumes of data. The Data Machine will have a total of 8 nodes, 4 focused on 
CPU-intensive jobs and 4 for jobs requiring GPUS.
 
The Data Machine will be connected to the existing HPCC file systems and compute resources. 
Though the CPU and GPU hardware is similar to what is offered by the current amd21 and 
amd22 clusters, what sets the Data Machine apart is the amount of memory available to CPU

* [similarity=0.483376] {'producer': 'pdfmake', 'total_pages': 5, 'creationdate': '2025-10-24T14:13:13+00:00', 'source': '/mnt/research/HPCC_AI_Workshop/use-case/data/icer_docs/data_machine.pdf', 'creator': 'pdfmake', 'page': 2, 'page_label': '3', 'start_index': 531} 

units, making them ideal for machine l

Now we'll delete the vector store, and re-set to the persistent vector store db. We should see a collection already there. Now we do not need to re-apply the embedding function to our reference docs.

In [21]:
del vector_store

In [22]:
vector_store = Chroma(
    collection_name="icer_docs_collection_full",
    embedding_function=embeddings,
    persist_directory="./chroma_icer_docs_db",
)
print(vector_store._collection.get(include=["embeddings"]))

{'ids': ['e9c426b6-9608-4130-8153-e54367b21d4f', '29951f21-1db2-451a-99ec-1309eaec9712', '6d572d7c-3b9d-4cef-9e38-1d58ffacefbc', '052b2362-5a5b-4510-a8cc-088ff74af7bb', 'c94d2aba-bf9b-4a9a-8667-0f89925edb60', '5ac15c5e-e5ae-435a-868d-c179b6448cbc', '727f76ed-56e0-467b-bb0a-8f8c830ffe73', 'f34cbef4-a499-4038-a945-e28823accc2d', '312bf843-1b2b-4a10-b140-0020d0c9806d', 'a20c788a-3497-46fc-a96e-80e0ef83d28a', 'bf65bffb-91fa-4f36-a3f1-77018b1c32d1', '42a68b30-e717-4d2f-a719-21d2ce6dfb80', 'd3793028-4e2e-406a-8c22-bce17808c7f6', '77eb5a60-624d-4bb6-8c22-3fce2b26d01f', '03de6633-511f-4541-9b17-f1acde22bfbd', '8f3ea9b9-e061-41c5-ad04-e287497ae9f3', 'fd8138db-95d2-4108-874a-eb0b5cc8743b', 'eebdb44e-a13b-405e-ba2d-bd9979fde515', '528bc380-c48f-40b5-9261-fda8f3ccee5a', '4615974f-c88c-453e-a122-030b2c26b4ea', 'd1fb34a1-3c9a-4598-a7c5-6a984dbec347', '7c8eb0a7-973c-48c8-8a48-8722ec13a28a', 'f05917f5-e6d4-4540-b7a1-4578c0dc5658', '54071259-2686-4505-b6d4-ae69682f3e84', '15c7b7fa-09c0-4452-8142-0aa85b

We may want to add additional docs to our vector store.

In [26]:
file_path = '/mnt/research/HPCC_AI_Workshop/use-case/data/icer_docs_2/LabNotebook_Update_HowtorunRAGinOpenWebUI.pdf'
docs = load_document(file_path) #in 'page' mode number of docs here == number of pages in the actual document

Loaded 3 docs from /mnt/research/HPCC_AI_Workshop/use-case/data/icer_docs_2/LabNotebook_Update_HowtorunRAGinOpenWebUI.pdf


In [27]:
new_splits = conditional_split(docs, chunk_size=700, chunk_overlap=100)
#splits = text_splitter.split_documents(all_docs)
print(f"Split into {len(new_splits)} chunks.")

Split into 8 chunks.


In [28]:
uuids = [str(uuid4()) for _ in range(len(new_splits))]
vector_store.add_documents(documents=new_splits, ids=uuids) # add_documents applies the embedding function to the new documents (new splits)

['be5cb32f-ec48-45f0-ac38-d1cacf042b41',
 '9e9997a9-c29f-4dce-9a5d-753ab8e6a77b',
 'c995d461-2d5d-469c-8a1a-d3936049c7f0',
 'c5a9a1bc-9ee0-4ddb-8af8-da98116ef118',
 '93df64f4-8a93-45bc-98b2-6a96fe48e438',
 'b4ba0932-2af3-4159-ac57-6583ab0bb91c',
 '697bfb5c-ff7b-46ea-a8d7-cb8128b3c432',
 'c70d6845-9c4e-4741-b0a9-b9695aae2701']

### Next steps

You can write a simple script to run an interactive chat with a docs chat assistant, which uses reference docs embedded in a vector store. You may use and/or reference the simple example we put in `scripts/docs_asst.py` and build on that, or test with your own set of reference docs. Note you can use the `--data_dir` flag to point the docs assistant to where your reference docs are stored, and it will apply an embedding on these docs. Note that if you do not provide specific files in that data directory, it will embed all docs. In either case, it will create these embeddings in-memory, and will not be persist between sessions (this is an example of something you could add). If you already have a persistent vector store directory (e.g., the chroma db you created in this notebook), you can simply point the docs assistant directly to the vector store using the `--persist_dir` flag. Another thing you may consider adding is the option to provide a persistent vector store db, as well as an additional set of docs which would be embedded and added to the existing (persistent) vector store. 

To run the `docs_asst.py` script with a full set of reference docs to be embedded, you would run  

`scripts/docs_asst.py --data_dir path/to/your/ref/docs`

To run the `docs_asst.py` script with a subset of reference docs to be embedded, you would run  

`scripts/docs_asst.py --data_dir path/to/your/ref/docs doc1.pdf doc2.pdf doc3.pdf`

To run the `docs_asst.py` script with a persistent vector store of your reference docs, you would run  

`scripts/docs_asst.py --persist_dir path/to/your/vector/store`

> Notes: (1) The LLM model for the docs assistant and embedding model for the vector store embeddings are hard coded as global variables in the `docs_asst.py` script as EMBED_MODEL and LLM_MODEL. You can modify these, or add a flag/option to the script to set your desired models accordingly. (2) Also, if using the `--persist_dir` flag it is currently setup and hard coded to use the `icer_docs_collection_full` collection that was created in this notebook. If you want to change this for a new set of reference docs you've embedded in a persistent vector store db, you can modify the script to use your new collection. 